In [16]:
# ============================================================
# MODELO FINAL CHURN EXPLICABLE - SIN TRAMPA / SIN LEAKAGE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

# ============================================================
# FEATURE ENGINEERING EXPLICABLE
# ============================================================

df_model["deterioro_total"] = (
    df_model["impagos_3m"]
    + df_model["stress_calidad_lag"]
    + df_model["media_retraso_3m"]
)

df_model["subida_factura_flag"] = (
    df_model["subida_factura"] > 0
).astype(int)

df_model["cliente_moroso"] = (
    (df_model["impagos_3m"] >= 2) &
    (df_model["media_retraso_3m"] >= 5)
).astype(int)

df_model["riesgo_financiero"] = (
    df_model["dias_retraso_pago"]
    + df_model["media_retraso_3m"]
    + (df_model["impagos_3m"] * 5)
)

df_model["riesgo_consumo"] = (
    (df_model["variacion_consumo_pct"] < 0)
).astype(int)

# ============================================================
# VARIABLES FINALES
# ============================================================

variables = [
    "stress_calidad_lag",
    "descuento_activo",
    "subida_factura",
    "subida_factura_flag",
    "variacion_consumo_pct",
    "cambio_consumo",
    "consumo_mes_anterior",
    "antiguedad_meses",
    "impago_flag",
    "impagos_3m",
    "media_retraso_3m",
    "dias_retraso_pago",
    "deterioro_total",
    "cliente_moroso",
    "riesgo_financiero",
    "riesgo_consumo"
]

# ============================================================
# LIMPIEZA
# ============================================================

df_temp = df_model[variables + ["churn"]].copy()
df_temp = df_temp.replace([np.inf, -np.inf], np.nan)
df_temp = df_temp.dropna()

X = df_temp[variables]
y = df_temp["churn"]

# ============================================================
# TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("="*60)
print("DISTRIBUCIÓN CHURN")
print("="*60)
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

# ============================================================
# MODELO
# ============================================================

modelo = Pipeline(steps=[
    ("smote", SMOTE(
        sampling_strategy=0.25,
        random_state=42,
        k_neighbors=5
    )),
    
    ("xgb", XGBClassifier(
        n_estimators=350,
        learning_rate=0.04,
        max_depth=4,
        min_child_weight=6,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=1,
        reg_alpha=1,
        reg_lambda=2,
        scale_pos_weight=1.5,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ))
])

modelo.fit(X_train, y_train)

# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]

# Threshold equilibrado
threshold = 0.28
y_pred = (y_prob >= threshold).astype(int)

# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "="*60)
print("RESULTADOS MODELO FINAL")
print("="*60)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Precision churn:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall churn:", round(recall_score(y_test, y_pred), 4))
print("F1 churn:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# IMPORTANCIA DE VARIABLES
# ============================================================

importancias = pd.DataFrame({
    "Variable": variables,
    "Importancia": modelo.named_steps["xgb"].feature_importances_
}).sort_values("Importancia", ascending=False)

print("\n" + "="*60)
print("IMPORTANCIA VARIABLES")
print("="*60)
print(importancias)

# ============================================================
# TABLA EXPLICABLE DE CLIENTES EN RIESGO
# ============================================================

clientes_riesgo = X_test.copy()
clientes_riesgo["churn_real"] = y_test.values
clientes_riesgo["probabilidad_churn"] = y_prob
clientes_riesgo["prediccion_churn"] = y_pred

clientes_riesgo = clientes_riesgo.sort_values(
    "probabilidad_churn",
    ascending=False
)

print("\n" + "="*60)
print("TOP 20 CLIENTES CON MAYOR RIESGO DE CHURN")
print("="*60)

print(
    clientes_riesgo[
        [
            "probabilidad_churn",
            "prediccion_churn",
            "churn_real",
            "stress_calidad_lag",
            "subida_factura",
            "variacion_consumo_pct",
            "impagos_3m",
            "media_retraso_3m",
            "riesgo_financiero",
            "deterioro_total"
        ]
    ].head(20)
)

DISTRIBUCIÓN CHURN
churn
0    17087
1     1186
Name: count, dtype: int64
churn
0    93.50955
1     6.49045
Name: proportion, dtype: float64

RESULTADOS MODELO FINAL
Accuracy: 80.08 %
Precision churn: 0.086
Recall churn: 0.2152
F1 churn: 0.1229
ROC-AUC: 0.6141

Matriz de confusión:
[[2876  542]
 [ 186   51]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      3418
           1       0.09      0.22      0.12       237

    accuracy                           0.80      3655
   macro avg       0.51      0.53      0.51      3655
weighted avg       0.88      0.80      0.84      3655


IMPORTANCIA VARIABLES
                 Variable  Importancia
0      stress_calidad_lag     0.183027
1        descuento_activo     0.120335
4   variacion_consumo_pct     0.095624
15         riesgo_consumo     0.093554
12        deterioro_total     0.071133
3     subida_factura_flag     0.064702
2          subida_factura     0.057847
5    